In [131]:
iata_to_icao = {
    "DEL": "VIDP",  # Delhi
    "BOM": "VABB",  # Mumbai
    "BLR": "VOBL",  # Bangalore
    "MAA": "VOMM",  # Chennai
    "HYD": "VOHS",  # Hyderabad
    "CCU": "VECC",  # Kolkata
    "AMD": "VAAH",  # Ahmedabad
    "COK": "VOCI",  # Cochin
    "GOI": "VOGO",  # Goa (Dabolim)
    "PNQ": "VAPO",  # Pune
    "LKO": "VILK",  # Lucknow
    "PAT": "VEPT",  # Patna
    "IXC": "VICG",  # Chandigarh
    "TRV": "VOTV",  # Trivandrum
    "IXM": "VOMD"   # Madurai
}

In [132]:
iata_airports = [
    "DEL","BOM","BLR","MAA","HYD","CCU","AMD",
    "COK","GOI","PNQ","LKO","PAT","IXC","TRV","IXM"
]

icao_airports = [iata_to_icao[iata] for iata in iata_airports]

print(icao_airports)

['VIDP', 'VABB', 'VOBL', 'VOMM', 'VOHS', 'VECC', 'VAAH', 'VOCI', 'VOGO', 'VAPO', 'VILK', 'VEPT', 'VICG', 'VOTV', 'VOMD']


In [92]:
import requests
import pandas as pd
import time
import csv

iata_airports = list(iata_to_icao.keys())
icao_airports = list(iata_to_icao.values())

# -------------------------------
# COLLECT DATA
# -------------------------------
airport_list = []

for iata, icao in zip(iata_airports, icao_airports):

    print(f"Fetching {iata} → {icao}")

    url = f"https://aerodatabox.p.rapidapi.com/airports/icao/{icao}"

    response = requests.get(url, headers=headers)

    if response.status_code == 200:

        data = response.json()

        # ---- Extract country ----
        country = data.get("country", {}).get("name")

        # ---- Extract continent ----
        continent = data.get("continent", {}).get("name")

        # ---- Fix timezone ----
        if isinstance(data.get("timeZone"), dict):
            timezone = data.get("timeZone", {}).get("name")
        else:
            timezone = data.get("timeZone")

        airport_record = {
            "icao_code": data.get("icao"),
            "iata_code": data.get("iata"),
            "name": data.get("fullName"),
            "city": data.get("municipalityName"),
            "country": country,
            "continent": continent,
            "latitude": data.get("location", {}).get("lat"),
            "longitude": data.get("location", {}).get("lon"),
            "timezone": timezone
        }

        airport_list.append(airport_record)

    else:
        print(f"❌ API Error for {icao}:", response.status_code)

    # Avoid rate limit
    time.sleep(1)

# -------------------------------
# DEBUG
# -------------------------------
print("Total airports fetched:", len(airport_list))

# -------------------------------
# SAVE CSV
# -------------------------------
with open("clean_airport_data.csv", "w", newline="", encoding="utf-8") as file:

    fieldnames = [
        "icao_code",
        "iata_code",
        "name",
        "city",
        "country",
        "continent",
        "latitude",
        "longitude",
        "timezone"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(airport_list)

print("✅ Airport data saved to CSV successfully!")

Fetching DEL → VIDP
Fetching BOM → VABB
Fetching BLR → VOBL
Fetching MAA → VOMM
Fetching HYD → VOHS
Fetching CCU → VECC
Fetching AMD → VAAH
Fetching COK → VOCI
Fetching GOI → VOGO
Fetching PNQ → VAPO
Fetching LKO → VILK
Fetching PAT → VEPT
Fetching IXC → VICG
Fetching TRV → VOTV
Fetching IXM → VOMD
Total airports fetched: 15
✅ Airport data saved to CSV successfully!


In [93]:
import requests
import pandas as pd
import uuid

# -------------------------------
# FUNCTION
# -------------------------------
def extract_flight(flight, airport_code, direction):

    departure = flight.get("departure", {})
    arrival = flight.get("arrival", {})
    aircraft = flight.get("aircraft", {})
    airline = flight.get("airline", {})

    # other airports
    dep_airport = (
        departure.get("airport", {}).get("iata") or
        departure.get("iata") or
        departure.get("airport", {}).get("icao")
    )

    arr_airport = (
        arrival.get("airport", {}).get("iata") or
        arrival.get("iata") or
        arrival.get("airport", {}).get("icao")
    )

    # direction logic
    if direction == "departure":
        origin = airport_code
        destination = arr_airport
    else:
        origin = dep_airport
        destination = airport_code

    # time fields
    sched_departure = (
        departure.get("scheduledTime", {}).get("local") or
        departure.get("scheduledTime", {}).get("utc") or "NA"
    )

    actual_departure = (
        departure.get("revisedTime", {}).get("local") or
        departure.get("actualTime", {}).get("local") or
        sched_departure
    )

    sched_arrival = (
        arrival.get("scheduledTime", {}).get("local") or
        arrival.get("scheduledTime", {}).get("utc") or "NA"
    )

    actual_arrival = (
        arrival.get("revisedTime", {}).get("local") or
        arrival.get("actualTime", {}).get("local") or
        sched_arrival
    )

    return {
        "flight_id": str(uuid.uuid4()),
        "flight_number": flight.get("number", "Unknown"),
        "aircraft_registration": aircraft.get("reg", "Unknown"),
        "origin_iata": origin or "Unknown",
        "destination_iata": destination or "Unknown",
        "scheduled_departure": sched_departure,
        "actual_departure": actual_departure,
        "scheduled_arrival": sched_arrival,
        "actual_arrival": actual_arrival,
        "status": flight.get("status", "Unknown"),
        "airline_code": airline.get("iata", "Unknown")
    }

# -------------------------------
# MAIN LOOP (MULTIPLE AIRPORTS)
# -------------------------------
all_records = []

for iata, icao in iata_to_icao.items():

    print(f"Fetching flights for {iata} ({icao})")

    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/2025-04-04T20:00/2025-04-05T08:00"

    querystring = {
        "withLeg": "true",
        "direction": "Both",
        "withCancelled": "true"
    }

    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code != 200:
        print(f"❌ Error for {icao}")
        continue

    flights = response.json()

    departures = flights.get("departures", [])
    arrivals = flights.get("arrivals", [])

    # departures
    for f in departures:
        all_records.append(extract_flight(f, iata, "departure"))

    # arrivals
    for f in arrivals:
        all_records.append(extract_flight(f, iata, "arrival"))

    time.sleep(1)  # avoid rate limit

# -------------------------------
# SAVE CSV
# -------------------------------
df_final = pd.DataFrame(all_records)

df_final.to_csv("cleanflight_data.csv", index=False)

print("✅ CSV SAVED SUCCESSFULLY")
print("Total rows:", len(df_final))

Fetching flights for DEL (VIDP)
Fetching flights for BOM (VABB)
Fetching flights for BLR (VOBL)
Fetching flights for MAA (VOMM)
Fetching flights for HYD (VOHS)
Fetching flights for CCU (VECC)
Fetching flights for AMD (VAAH)
Fetching flights for COK (VOCI)
Fetching flights for GOI (VOGO)
Fetching flights for PNQ (VAPO)
Fetching flights for LKO (VILK)
Fetching flights for PAT (VEPT)
Fetching flights for IXC (VICG)
Fetching flights for TRV (VOTV)
Fetching flights for IXM (VOMD)
✅ CSV SAVED SUCCESSFULLY
Total rows: 2342


In [104]:
import pandas as pd
import numpy as np
from datetime import datetime



df_flights = pd.DataFrame(all_records)

# Prepare the delay table
delay_records = []
delay_id = 1

for airport_iata in iata_to_icao.keys():

    # Filter flights for this airport (either origin or destination)
    df_airport = df_flights[(df_flights['origin_iata'] == airport_iata) |
                             (df_flights['destination_iata'] == airport_iata)]

    total_flights = len(df_airport)
    delayed_flights = 0
    canceled_flights = 0
    delays = []

    for _, row in df_airport.iterrows():
        # Check cancelled
        status = row.get("status", "")
        if pd.notna(status) and "cancel" in status.lower():
            canceled_flights += 1
            continue

        # Calculate delay in minutes
        sched = row.get("scheduled_departure")
        actual = row.get("actual_departure")

        if pd.notna(sched) and pd.notna(actual):
            try:
                sched_dt = datetime.fromisoformat(sched)
                actual_dt = datetime.fromisoformat(actual)
                delay_min = (actual_dt - sched_dt).total_seconds() / 60

                if delay_min > 0:
                    delays.append(delay_min)
                    delayed_flights += 1

            except:
                continue

    avg_delay = int(np.mean(delays)) if delays else 0
    median_delay = int(np.median(delays)) if delays else 0

    delay_records.append({
        "delay_id": delay_id,
        "airport_iata": airport_iata,
        "delay_date": "2025-04-04",  # You can change per actual date
        "total_flights": total_flights,
        "delayed_flights": delayed_flights,
        "avg_delay_min": avg_delay,
        "median_delay_min": median_delay,
        "canceled_flights": canceled_flights
    })

    delay_id += 1

# Save CSV
df_delay = pd.DataFrame(delay_records)
df_delay.to_csv("cleandelay_data.csv", index=False)
print("✅ Delay data saved")
print(df_delay)
print("Total rows:", len(df_final))

✅ Delay data saved
    delay_id airport_iata  delay_date  total_flights  delayed_flights  \
0          1          DEL  2025-04-04            752              119   
1          2          BOM  2025-04-04            493               37   
2          3          BLR  2025-04-04            513               82   
3          4          MAA  2025-04-04            313               10   
4          5          HYD  2025-04-04            343               14   
5          6          CCU  2025-04-04            227               13   
6          7          AMD  2025-04-04            198               33   
7          8          COK  2025-04-04            142                8   
8          9          GOI  2025-04-04             79                1   
9         10          PNQ  2025-04-04            179                5   
10        11          LKO  2025-04-04            110                4   
11        12          PAT  2025-04-04             44                1   
12        13          IXC  2025-

In [133]:
import requests
import csv
import time
import json


# -------------------------------
# Prepare Aircraft List
# -------------------------------
aircraft_list = []
seen_registration = set()
aircraft_id = 1

# -------------------------------
# Fetch flights for each airport
# -------------------------------
for iata, icao in iata_to_icao.items():
    print(f"Fetching flights for {iata} ({icao})")

    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/2025-04-04T20:00/2025-04-05T08:00"
    querystring = {"withLeg": "true", "direction": "Both", "withCancelled": "true"}

    response = requests.get(url, headers=headers, params=querystring)
    
    if response.status_code != 200:
        print(f"❌ Error fetching {icao}: {response.status_code}")
        continue

    flights_data = response.json()
    flights = flights_data.get("departures", []) + flights_data.get("arrivals", [])

    # Convert any string items to dict
    clean_flights = []
    for f in flights:
        if isinstance(f, str):
            try:
                clean_flights.append(json.loads(f))
            except json.JSONDecodeError:
                continue
        elif isinstance(f, dict):
            clean_flights.append(f)

    # -------------------------------
    # Extract Aircraft Data
    # -------------------------------
    for flight in clean_flights:
        aircraft = flight.get("aircraft", {})
        registration = aircraft.get("reg") or flight.get("aircraft_registration")
        if not registration or registration in seen_registration:
            continue

        model = aircraft.get("model")
        manufacturer = model.split()[0] if model else None
        icao_type_code = aircraft.get("icaoType") or aircraft.get("type", {}).get("icaoCode")
        if not icao_type_code and model:
            for w in model.split():
                if any(char.isdigit() for char in w):
                    icao_type_code = w
                    break

        owner = flight.get("airline", {}).get("name") or flight.get("airline_code")

        aircraft_list.append([aircraft_id, registration, model, manufacturer, icao_type_code, owner])
        seen_registration.add(registration)
        aircraft_id += 1

    time.sleep(1)  # avoid rate limits

# -------------------------------
# Save to CSV
# -------------------------------
csv_file = "aircrafts_data.csv"
with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["aircraft_id","registration","model","manufacturer","icao_type_code","owner"])
    writer.writerows(aircraft_list)

print(f"✅ Aircraft data saved to {csv_file}")
print("Total Aircraft Collected:", len(aircraft_list))

Fetching flights for DEL (VIDP)
Fetching flights for BOM (VABB)
Fetching flights for BLR (VOBL)
Fetching flights for MAA (VOMM)
Fetching flights for HYD (VOHS)
Fetching flights for CCU (VECC)
Fetching flights for AMD (VAAH)
Fetching flights for COK (VOCI)
Fetching flights for GOI (VOGO)
Fetching flights for PNQ (VAPO)
Fetching flights for LKO (VILK)
Fetching flights for PAT (VEPT)
Fetching flights for IXC (VICG)
Fetching flights for TRV (VOTV)
Fetching flights for IXM (VOMD)
✅ Aircraft data saved to aircrafts_data.csv
Total Aircraft Collected: 528


In [1]:
import pandas as pd

# =========================
# 1. Load Data
# =========================
df = pd.read_csv("cleanflight_data.csv")

# =========================
# 2. Standardize Column Names
# =========================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# =========================
# 3. Remove Duplicates
# =========================
df = df.drop_duplicates()

# =========================
# 4. Handle Missing Values
# =========================

# Drop rows where all values are missing
df = df.dropna(how='all')

# Fill categorical columns with 'Unknown'
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

# Fill numeric columns with median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# =========================
# 5. Convert Date Columns (if exist)
# =========================
date_cols = ['departure_time', 'arrival_time', 'scheduled_departure']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# =========================
# 6. Remove Invalid Values
# =========================

# Example: remove negative delays
if 'departure_delay' in df.columns:
    df = df[df['departure_delay'] >= 0]

if 'arrival_delay' in df.columns:
    df = df[df['arrival_delay'] >= 0]

# =========================
# 7. Standardize Text Data
# =========================
if 'status' in df.columns:
    df['status'] = df['status'].str.title().str.strip()

if 'airline_code' in df.columns:
    df['airline_code'] = df['airline_code'].str.upper().str.strip()

# =========================
# 8. Feature Engineering (Optional but useful)
# =========================
if 'departure_delay' in df.columns:
    df['delay_category'] = pd.cut(
        df['departure_delay'],
        bins=[-1, 15, 60, float('inf')],
        labels=['On Time', 'Moderate Delay', 'High Delay']
    )

# =========================
# 9. Save Cleaned Data
# =========================
df.to_csv("cleaned_flight_data.csv", index=False)

print("✅ Data cleaning completed successfully!")
print(df.head())

✅ Data cleaning completed successfully!
                              flight_id flight_number aircraft_registration  \
0  477d0852-a92e-4839-9f5c-385e8c289eef       6E 2163               Unknown   
1  23f13d40-839b-491b-895e-49afbfffe3d8       AI 2637               Unknown   
2  b2cc37a5-9a60-4497-a67c-d46b063c8a93       AI 2835                VT-TQG   
3  7d1a6153-d3d3-47fb-a420-181fb98d5df7        SG 477                VT-SGG   
4  1cb48d5d-a8f2-40dc-b044-6fe7f9ffd062        SV 759               Unknown   

  origin_iata destination_iata        scheduled_departure  \
0         DEL              BOM  2025-04-04 20:00:00+05:30   
1         DEL              IXC  2025-04-04 20:00:00+05:30   
2         DEL              MAA  2025-04-04 20:00:00+05:30   
3         DEL          Unknown  2025-04-04 15:25:00+05:30   
4         DEL              JED  2025-04-04 20:05:00+05:30   

         actual_departure       scheduled_arrival          actual_arrival  \
0  2025-04-04 20:00+05:30  2025-04-04 22:

C:\Users\HP\AppData\Local\Temp\ipykernel_105672\3527584791.py:45: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors='coerce')
